# HydraY NNUE - layer intermedio: 1024 → 16 → 1

Runtime → **GPU (T4)**, poi esegui le celle in ordine. ~4h in **otto tappe**.

### La domanda
Fin qui la rete e' un solo strato: dall'accumulatore da 1024 si va dritti alla
valutazione. Qui in mezzo compare **un layer da 16**, e con lui il prodotto a
coppie (*pairwise*) che dimezza 1024 a 512 per prospettiva e fornisce la
nonlinearita' quadratica.

**Cambia SOLO l'architettura.** Stesso dataset v7, stesso budget 160 superbatch,
stesse quattro fette nello stesso ordine, stesso WDL, stesso learning rate. Se
questa rete vince, e' merito del layer; se perde, e' colpa sua.

### Cosa viene in blocco col layer
Due cose non sono libere di restare come prima, fanno parte del pacchetto:

1. **CReLU al posto di SCReLU** sul feature transformer - la nonlinearita'
   quadratica ora la da' il pairwise;
2. **init con fan-in 32** (`init_with_effective_input_size`) - l'init giusto per
   ingressi sparsi, che su una rete piu' profonda puo' decidere fra addestrarsi
   e non addestrarsi affatto.

Sono inseparabili dall'esperimento: un risultato negativo non potra' distinguere
fra le tre cause. E' il prezzo accettato per non fare tre run.

### Un layer solo, non due
L'esempio di bullet ne usa due (16 → 32 → 1). Il secondo costa pochissimo - 544
moltiplicazioni contro 16.384 - ma e' un elemento in piu' non validato su una
rete che e' un terzo di quelle per cui quella proporzione e' stata trovata. Con
gia' tre cambiamenti inseparabili, un quarto allungherebbe solo la lista dei
sospetti.

### Il costo e' gia' pagato e misurato
Il forward C++ e' scritto, verificato bit per bit contro l'oracolo Rust e
ottimizzato: **2,15x** una valutazione a un layer, non 2,92x come nella prima
stesura. In motore vale circa **−12% di NPS**. Quindi il layer parte in debito:
deve rendere piu' di quanto costa in nodi al secondo.

### Il controllo che conta, e va fatto SUBITO
Dopo la **prima tappa** (superbatch 20) guarda la `running loss`. La rete a un
layer, stesso dataset e stesso punto, stava intorno a **0,0134**. Se qui e'
molto piu' alta o non scende, l'architettura non sta imparando e conviene
fermarsi dopo mezz'ora invece che dopo quattro ore.


In [ ]:
# --- helper: qualunque comando fallito ferma il notebook, e l'output si vede ---
import subprocess, os, sys, json

def sh(cmd):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'FALLITO (exit {p.returncode}): {cmd}')

sh('nvidia-smi --query-gpu=name,memory.total --format=csv')
sh('df -h /content | tail -1')
print('\nGPU presente. Se la riga sopra non mostra una T4, cambia runtime.')

In [ ]:
# --- Drive + configurazione ---
from google.colab import drive
drive.mount('/content/drive')

import glob
def find(name):
    hits = glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=True)
    assert hits, f'{name} non trovato su Drive'
    return hits[0]

PARTS = {i: find(f'hydray_v7_part{i}.bin.zst') for i in (1, 2, 3, 4)}
# Taglia attesa DOPO la decompressione, per parte. Una decompressione
# interrotta a meta' produce un file piu' corto e nessun errore: senza questo
# controllo si addestrerebbe in silenzio su dati troncati.
RAW_SIZE = {1: 23_742_906_368, 2: 23_742_906_368,
            3: 23_742_906_368, 4: 23_745_983_264}
for i, p in PARTS.items():
    print(f'parte {i}: {os.path.getsize(p)/2**30:6.2f} GiB compressa  {p}')

NET_ID   = 'hydray-deep16-160sb'
TOTAL_SB = 160          # stesso budget e stessi dati della rete adottata:
                        # l'unica variabile e' l'architettura
STAGE    = 20           # otto tappe
ORDER    = [1, 2, 3, 4, 1, 2, 3, 4]   # ogni fetta girata due volte
TRAINER  = '/content/th/nnue/trainer'
assert len(ORDER) * STAGE == TOTAL_SB and STAGE % 10 == 0

In [ ]:
# --- Rust ---
sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
sh('$HOME/.cargo/bin/cargo --version')

In [ ]:
# --- clone + verifica architettura ---
# La verifica non e' cerimoniale: un run precedente ha addestrato per un'ora
# un'architettura diversa da quella creduta, perche' il clone era sbagliato e il
# nome del checkpoint non dice nulla sul contenuto.
#
# ⚠️ BRANCH = 'dev', NON 'nnue-1024': trainer_deep.rs e sanity_deep.rs esistono
# solo su dev. Su nnue-1024 il clone riesce e il cargo run fallisce dopo.
BRANCH = 'dev'
sh('rm -rf /content/th')
sh(f'git clone --depth 1 --branch {BRANCH} https://github.com/ThomasGhione/HydraY /content/th')

tr = open(f'{TRAINER}/src/bin/trainer_deep.rs').read()
assert 'const HIDDEN_SIZE: usize = 1024;' in tr, 'trainer_deep.rs non e a 1024'
assert 'const L1_SIZE: usize = 16;' in tr, 'il layer intermedio non e da 16'
assert 'const OUTPUT_BUCKETS: usize = 8;' in tr, 'gli output bucket devono restare 8'
sd = open(f'{TRAINER}/src/bin/sanity_deep.rs').read()
assert 'const HIDDEN: usize = 1024;' in sd, 'sanity_deep.rs non e a 1024'
assert 'const L1_SIZE: usize = 16;' in sd, 'sanity_deep.rs non ha il layer da 16'
assert 'const INPUT_BUCKETS: usize = 4;' in sd, 'i king bucket devono restare 4'
print(f'branch {BRANCH}, 1024 -> 16 -> 1, 4 king bucket, 8 output bucket: ok')

sh('apt-get -qq install -y zstd >/dev/null')
st = os.statvfs('/content'); free_gb = st.f_bavail * st.f_frsize / 2**30
print(f'liberi {free_gb:.1f} GiB, picco atteso ~36 GiB (una fetta + cache di Drive)')
assert free_gb > 45, 'disco insufficiente'


In [ ]:
# --- helper delle tappe (ESEGUIRE SEMPRE, anche in ripartenza) ---

def load_slice(n):
    """Scompatta la fetta n in /content/data.bin, sostituendo la precedente."""
    if os.path.exists('/content/data.bin'):
        os.remove('/content/data.bin')          # spazio prima, non dopo
    sh(f'zstd -d -T0 --long=27 -c "{PARTS[n]}" > /content/data.bin')
    got = os.path.getsize('/content/data.bin')
    assert got == RAW_SIZE[n], f'fetta {n} troncata: {got} != {RAW_SIZE[n]}'
    print(f'fetta {n}: {got//32/1e6:.1f}M posizioni, taglia verificata', flush=True)

def stage_cmd(end, start, resume_from):
    """⚠️ STAGE_END DEVE STARE ATTACCATO A `cargo`, non in testa alla riga.
    `STAGE_END=40 cd dir && cargo ...` assegna la variabile SOLO a `cd`: cargo
    la riceve vuota, il trainer ignora le tappe e tira dritto fino a TOTAL_SB
    senza salvare niente. E' costato un run intero. Da qui l'`env` esplicito."""
    args = f'/content/data.bin {TOTAL_SB} {NET_ID}'
    if resume_from is not None:
        args += f' {start} checkpoints/{NET_ID}-{resume_from}'
    return (f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH '
            f'env CUDA_PATH=/usr/local/cuda STAGE_END={end} '
            f'cargo run -r --bin trainer_deep --features cuda -- {args}')

def save_to_drive(end):
    """Copia il checkpoint su Drive e VERIFICA che ci sia arrivato davvero: il
    mount di Drive scrive attraverso una cache, quindi un upload mai completato
    passerebbe per riuscito."""
    ck  = f'{TRAINER}/checkpoints/{NET_ID}-{end}'
    dst = f'/content/drive/MyDrive/{NET_ID}-{end}'
    assert os.path.isdir(ck), f'checkpoint mancante in locale: {ck}'
    sh(f'rm -rf {dst} && cp -r {ck} /content/drive/MyDrive/')
    size = lambda p: sum(os.path.getsize(os.path.join(d, f))
                         for d, _, fs in os.walk(p) for f in fs)
    assert os.path.isdir(dst), f'la copia su Drive non esiste: {dst}'
    assert size(dst) == size(ck), f'copia su Drive incompleta: {size(dst)} != {size(ck)}'
    print(f'tappa fino al superbatch {end} su Drive ({size(dst)/2**20:.0f} MiB, verificata)', flush=True)

def run_stages(done=0):
    """Esegue le tappe da `done` in poi. done=0 parte da zero."""
    assert done % STAGE == 0, f'{done} non e un confine di tappa'
    prev = done if done else None
    for k in range(done // STAGE, len(ORDER)):
        end, start, sl = (k+1)*STAGE, k*STAGE + 1, ORDER[k]
        print(f'\n===== tappa {k+1}/{len(ORDER)}: superbatch {start}-{end}, fetta {sl} =====', flush=True)
        load_slice(sl)
        sh(stage_cmd(end, start, prev))
        save_to_drive(end)
        prev = end

In [ ]:
# --- training: otto tappe, fette 1-2-3-4-1-2-3-4 ---
# NON eseguire questa cella in una ripartenza: usa invece la cella in fondo.
run_stages(done=0)

In [ ]:
# --- verifica finale e salvataggio su Drive ---
final = f'{TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB}/quantised.bin'
sz = os.path.getsize(final)
# 6.425.632 di payload, arrotondati a 64. La rete a UN layer ne fa 6.326.336:
# se esce quel numero, e' stato eseguito il trainer sbagliato.
assert 6425632 <= sz < 6425632 + 64, f'taglia {sz}: NON e la rete 1024->16->1'
print('quantised.bin:', sz, 'byte - layer intermedio da 16 confermato\n')

sh(f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH cargo run -r --bin sanity_deep -- {final}')
sh(f'cp -r {TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB} /content/drive/MyDrive/')
print('\n' + '='*70)
print('RIFERIMENTO - la rete adottata (v7, un layer, 160 SB), misurata in locale:')
print('  startpos            48      mediogioco ~24 pezzi   930')
print('  KQvK               929      KRPvKR                 110')
print('  cavallo in piu     730      re attivi (finale)     116')
print('  donna in piu      1824      training loss     0.012734')
print()
print('⚠️ La loss NON e confrontabile fra le due architetture: il pairwise')
print('cambia la funzione, non solo i pesi. Serve solo a vedere se scende.')
print('I sanity si guardano per accorgersi di un disastro, non per prevedere')
print('l Elo: la rete a 80 SB aveva KQvK fermo e vinse di +29,4.')
print('Il verdetto e lo SPRT, e basta.')
print('='*70)


In [ ]:
# --- RIPARTENZA (usare SOLO se la sessione e' morta a meta') ---
# Come si usa:
#   1. esegui le celle da "helper" fino a "helper delle tappe" compresa;
#   2. NON eseguire la cella del training;
#   3. metti RESUME = True e DONE = ultimo superbatch salvato su Drive.
# La fetta giusta viene ricavata da ORDER: non devi ricordarti dov'era.
#
# Con RESUME = False questa cella non fa niente, cosi' "Esegui tutte" e' sicuro
# (altrimenti, a run finito, ripartirebbe da DONE rifacendo ore di training).

RESUME = False
DONE   = 20      # ultimo superbatch salvato su Drive

if not RESUME:
    print('ripartenza disattivata (RESUME = False) - nessuna azione')
else:
    ck = f'/content/drive/MyDrive/{NET_ID}-{DONE}'
    assert os.path.isdir(ck), f'checkpoint non trovato su Drive: {ck}'
    os.makedirs(f'{TRAINER}/checkpoints', exist_ok=True)
    sh(f'cp -r {ck} {TRAINER}/checkpoints/')
    assert os.path.isdir(f'{TRAINER}/checkpoints/{NET_ID}-{DONE}')
    print(f'ripartenza dal superbatch {DONE} (prossima fetta: {ORDER[DONE//STAGE]})\n')
    run_stages(done=DONE)

## Come leggere il risultato

Lo SPRT e' testa a testa contro la rete adottata (v7, un layer, 160 SB), stesso
binario da entrambe le parti, candidata via `EvalFile`. Ma attenzione: qui
**non e' solo un cambio di pesi**. Il binario riconosce la rete profonda dalla
taglia del file e cambia percorso di valutazione, quindi la candidata gioca
anche con **~12% di NPS in meno**. Il numero che esce e' gia' al netto del
costo: e' il guadagno vero, non quello lordo dell'architettura.

**Se vince** - il layer paga anche pagandosi i nodi persi, e allora diventa
sensato guardare le due strade che oggi sono chiuse: il secondo layer (16 → 32
→ 1) e soprattutto la **sparsita'**, che nel forward vale forse 2-3x sul layer
l1 e restituirebbe gran parte del NPS perduto.

**Se pareggia** - l'architettura guadagna quanto il NPS perde. In quel caso la
mossa giusta non e' buttarla: e' recuperare il NPS con la sparsita' e rimisurare,
perche' il lordo era positivo.

**Se perde nettamente** - o l'architettura non rende su una rete da 1024, o le
5,4 epoche non bastano a un modello con piu' parametri. Non si potra' distinguere
senza un secondo run a budget doppio.

### Una cosa da non rifare
I sanity eval **non predicono l'Elo**. La rete a 80 superbatch aveva KQvK fermo
a 657 e sembrava a corto di dati; ha poi vinto di +29,4. Guardali per accorgerti
di un disastro - mirror rotto, valori assurdi, taglia sbagliata - non per
prevedere il risultato.

### E la loss della prima tappa?
Quella si', va guardata subito: non per prevedere l'Elo, ma per capire se
l'addestramento e' partito. Vedi la nota in cima al notebook.
